# Análisis de resultados: Gemini para texto e imágenes

Este notebook compara las predicciones guardadas por los dos scripts con las referencias anotadas. No llama a Gemini ni inventa datos: si todavía no ejecuté los scripts, me indica qué archivos faltan.

In [ ]:
import json
import math
import unicodedata
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'reto1.py').is_file()), Path.cwd())
OUTPUT_DIR = ROOT / 'outputs'

def load_results(filename):
    path = OUTPUT_DIR / filename
    if not path.is_file():
        print(f'Falta {path.relative_to(ROOT)}. Ejecuta el script correspondiente desde la carpeta del proyecto.')
        return []
    with path.open(encoding='utf-8') as result_file:
        data = json.load(result_file)
    if not isinstance(data, list):
        raise ValueError(f'{filename} debe contener una lista JSON.')
    return data

text_results = load_results('text_results.json')
image_results = load_results('image_results.json')
print(f'Resultados cargados: {len(text_results)} textos y {len(image_results)} imágenes.')

## Métricas

La accuracy mide aciertos sobre ejemplos evaluables. Precision, recall y F1 se calculan de forma micro para entidades y objetos, comparando nombres normalizados por ejemplo. Las referencias visuales nulas se excluyen; una cadena OCR vacía sí cuenta como referencia válida.

In [ ]:
def normalize(value):
    text = unicodedata.normalize('NFKC', str(value)).casefold()
    return ' '.join(text.split())

def add_metric(rows, task, metric, value, count):
    rows.append({'Tarea': task, 'Métrica': metric, 'Valor': value, 'Ejemplos evaluados': count})

def accuracy(rows, task, pairs):
    valid = [(expected, predicted) for expected, predicted in pairs if expected is not None and predicted is not None]
    value = sum(normalize(expected) == normalize(predicted) for expected, predicted in valid) / len(valid) if valid else math.nan
    add_metric(rows, task, 'accuracy', value, len(valid))

def macro_classification_prf(rows, task, pairs):
    valid = [(normalize(expected), normalize(predicted)) for expected, predicted in pairs if expected is not None and predicted is not None]
    labels = {label for pair in valid for label in pair}
    if not valid or not labels:
        return
    scores = []
    for label in labels:
        true_positive = sum(expected == label and predicted == label for expected, predicted in valid)
        false_positive = sum(expected != label and predicted == label for expected, predicted in valid)
        false_negative = sum(expected == label and predicted != label for expected, predicted in valid)
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
        scores.append((precision, recall, f1))
    count = len(valid)
    for index, metric in enumerate(('precision macro', 'recall macro', 'F1 macro')):
        add_metric(rows, task, metric, sum(score[index] for score in scores) / len(scores), count)

def micro_prf(rows, task, reference_key, predicted_key, records):
    true_positive = false_positive = false_negative = 0
    evaluated = 0

    def normalized_names(items):
        names = set()
        for item in items:
            value = item.get('name', '') if isinstance(item, dict) else item
            if str(value).strip():
                names.add(normalize(value))
        return names

    for record in records:
        prediction = record.get('prediction')
        if prediction is None:
            continue
        expected = record.get(reference_key)
        if expected is None:
            continue
        expected_values = normalized_names(expected)
        predicted_values = normalized_names(prediction.get(predicted_key, []))
        true_positive += len(expected_values & predicted_values)
        false_positive += len(predicted_values - expected_values)
        false_negative += len(expected_values - predicted_values)
        evaluated += 1
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else math.nan
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else math.nan
    f1 = 2 * precision * recall / (precision + recall) if not math.isnan(precision) and not math.isnan(recall) and precision + recall else math.nan
    add_metric(rows, task, 'precision micro', precision, evaluated)
    add_metric(rows, task, 'recall micro', recall, evaluated)
    add_metric(rows, task, 'F1 micro', f1, evaluated)

metrics = []
valid_texts = [item for item in text_results if item.get('prediction') is not None]
language_pairs = [(item.get('language_reference'), item['prediction'].get('language')) for item in valid_texts]
sentiment_pairs = [(item.get('sentiment_reference'), item['prediction'].get('sentiment')) for item in valid_texts]
accuracy(metrics, 'Idioma', language_pairs)
accuracy(metrics, 'Sentimiento', sentiment_pairs)
macro_classification_prf(metrics, 'Sentimiento', sentiment_pairs)
micro_prf(metrics, 'Entidades nombradas', 'entities_reference', 'entities', valid_texts)

valid_images = [item for item in image_results if item.get('prediction') is not None]
category_pairs = [(item.get('expected_scene_category'), item['prediction'].get('scene_category')) for item in valid_images]
accuracy(metrics, 'Clasificación de escena', category_pairs)
macro_classification_prf(metrics, 'Clasificación de escena', category_pairs)
accuracy(metrics, 'OCR exacto', [(item.get('expected_ocr'), item['prediction'].get('ocr_text')) for item in valid_images if item.get('expected_ocr') is not None])
micro_prf(metrics, 'Detección de objetos', 'expected_objects', 'objects', valid_images)

metric_table = metrics
if not metric_table:
    print('Todavía no hay métricas calculables. Genera resultados y completa las referencias visuales.')
else:
    print(f"{'Tarea':<28} {'Métrica':<20} {'Valor':>8} {'Ejemplos':>10}")
    for row in metric_table:
        value = f"{row['Valor']:.3f}" if not math.isnan(row['Valor']) else 'n/d'
        print(f"{row['Tarea']:<28} {row['Métrica']:<20} {value:>8} {row['Ejemplos evaluados']:>10}")

## Comparación visual

El gráfico facilita comparar las métricas disponibles en la misma escala. Las tareas de OCR y objetos solo aparecen cuando completé las referencias correspondientes.

In [ ]:
if not metric_table:
    print('Todavía no hay métricas calculables. Genera resultados y completa las referencias visuales.')
else:
    plot_data = [row for row in metric_table if not math.isnan(row['Valor'])]
    if not plot_data:
        print('No hay puntuaciones numéricas para graficar.')
    elif plt is None:
        print('Las métricas están calculadas. Instala matplotlib con "python -m pip install matplotlib" para ver el gráfico.')
    else:
        labels = [f"{row['Tarea']} · {row['Métrica']}" for row in plot_data]
        values = [row['Valor'] for row in plot_data]
        figure, axis = plt.subplots(figsize=(9, max(3, len(plot_data) * 0.42)))
        axis.barh(labels, values, color='#167d8d')
        axis.set_xlim(0, 1)
        axis.set_xlabel('Puntuación (0 a 1)')
        axis.set_ylabel('')
        axis.set_title('Resultados sobre los ejemplos anotados')
        axis.grid(axis='x', alpha=0.25)
        figure.tight_layout()
        plt.show()

## Interpretación

Interpreto estos resultados como una comparación didáctica sobre diez textos y cinco imágenes, no como una medida general del rendimiento del modelo. Antes de sacar conclusiones reviso los errores de cada ejemplo y confirmo que las etiquetas de referencia sean correctas. Si una referencia de OCR u objetos está vacía (`null`), esa tarea no se evalúa para esa imagen.